# MercaRAG - Notebook de arranque (Docker)
Verifica que el entorno de Docker funciona: conexión a PostgreSQL con pgvector, esquema cargado e índices vectoriales.

Las librerías ya vienen instaladas en la imagen de Jupyter, así que no hace falta `pip install`.

# 1. Importaciones y preparación

In [1]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import psycopg2
from psycopg2.extras import Json
from pgvector.psycopg2 import register_vector

from sentence_transformers import SentenceTransformer
print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


# 2. Conexión a PostgreSQL (contenedor `db`)
La URL viene de la variable de entorno `DATABASE_URL`, definida en `docker-compose.yml`.
Dentro de la red de Docker el servidor se llama `db` (no `localhost`).

In [2]:
DATABASE_URL = os.environ["DATABASE_URL"]

def get_connection():
    connection = psycopg2.connect(DATABASE_URL)
    register_vector(connection)
    return connection

connection = get_connection()
with connection.cursor() as cur:
    cur.execute("SELECT version()")
    print(cur.fetchone()[0])
connection.close()
print("Conexión exitosa con la base de datos.")

PostgreSQL 17.11 (Debian 17.11-1.pgdg12+2) on x86_64-pc-linux-gnu, compiled by gcc (Debian 12.2.0-14+deb12u1) 12.2.0, 64-bit
Conexión exitosa con la base de datos.


# 3. Verificación del esquema

In [3]:
connection = get_connection()
try:
    with connection.cursor() as cur:
        cur.execute("""SELECT table_name FROM information_schema.tables
                       WHERE table_schema = 'public' AND table_type = 'BASE TABLE'
                       ORDER BY table_name""")
        tablas = [fila[0] for fila in cur.fetchall()]
        print(f"Tablas en la base: {len(tablas)}")
        print(", ".join(tablas))

        cur.execute("""SELECT indexname, tablename FROM pg_indexes
                       WHERE schemaname = 'public' AND indexdef ILIKE '%hnsw%'
                       ORDER BY indexname""")
        print("\nÍndices vectoriales (HNSW):")
        for nombre, tabla in cur.fetchall():
            print(f"  {nombre}  ->  {tabla}")
finally:
    connection.close()

Tablas en la base: 27
administrativo, categoria, cliente, compra, conductor, consulta, detalle_compra, direccion, documento, domicilio, embeddings_imagen, embeddings_texto, empleado, evaluacion, imagen, imagen_etiqueta, inventario, marca, producto, promocion, promocion_producto, proveedor, punto_venta, query_embeddings, resultado, usuario, vehiculo

Índices vectoriales (HNSW):
  idx_emb_imagen_hnsw  ->  embeddings_imagen
  idx_emb_texto_hnsw_fijo  ->  embeddings_texto
  idx_emb_texto_hnsw_oraciones  ->  embeddings_texto
  idx_emb_texto_hnsw_semantico  ->  embeddings_texto
